[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/main/notebooks/case_studies/service_assistant/langgraph.ipynb)

# Simulated service assistant — LangGraph

**Task.** Process a user-specified simulated service request under an exact, pre-authorised action.

**Conceptual stages.** `inspect_state → propose_action → authorise_action → checkpoint → execute_action → verify_effect → confirm`.

The model may propose and confirm, but deterministic Python owns approval, state changes, recovery and duplicate suppression.


In [ ]:
import os

# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "service-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = os.getenv(
    "AGENTIC_TUTORIAL_LOCAL_MODEL_PATH",
    "models/local/Qwen3-0.6B-Q8_0.gguf",
)
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

SERVICE_REQUEST = "Update the delivery address for order ord-100 to 2 Evidence Road."
APPROVED_ORDER_ID = "ord-100"
APPROVED_NEW_ADDRESS = "2 Evidence Road"
APPROVED_IDEMPOTENCY_KEY = "address-ord-100-v1"
ACTOR = "tutorial-user"

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "langgraph==1.2.9", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import json
from pathlib import Path
from typing import ClassVar, Literal

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ConfigDict

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import SimulatedService  # noqa: E402

service_path = ROOT / "data/service_assistant/simulated_service.json"
fixture_path = ROOT / "data/service_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Matched workflow

The seven conceptual stages match the paper. All effects remain local and simulated; the approval gate compares the complete proposed action with the explicit approval.


In [ ]:
# --- Matched case-study implementation ---


class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Action(Strict):
    schema_id: ClassVar[str] = "service.action.v1"
    action: Literal["update_address"]
    order_id: str
    new_address: str
    idempotency_key: str
    requires_approval: bool


class Confirmation(Strict):
    schema_id: ClassVar[str] = "service.confirmation.v1"
    message: str
    status: Literal["completed"]


Action.model_rebuild(_types_namespace={"Literal": Literal})
Confirmation.model_rebuild(_types_namespace={"Literal": Literal})


def create_client():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


async def propose(client, schema, text):
    response = await client.generate([user(text)], response_schema=schema)
    return schema.model_validate(response.structured_output)


def inspect_state(service):
    order = service.read_order(APPROVED_ORDER_ID, actor=ACTOR)
    return order


async def propose_action(client, request):
    return await propose(
        client,
        Action,
        (
            f"User request: {request!r}. Propose one typed action. "
            f"Use idempotency key {APPROVED_IDEMPOTENCY_KEY!r} and mark it as requiring approval. "
            "Do not claim that the action was executed."
        ),
    )


def authorise_action(action):
    approved = {
        "action": "update_address",
        "order_id": APPROVED_ORDER_ID,
        "new_address": APPROVED_NEW_ADDRESS,
        "idempotency_key": APPROVED_IDEMPOTENCY_KEY,
        "requires_approval": True,
    }
    return action.model_dump() == approved


def checkpoint(service):
    return SimulatedService.resume(service.checkpoint())


def execute_action(service, action):
    return service.update_address(
        action.order_id,
        action.new_address,
        actor=ACTOR,
        idempotency_key=action.idempotency_key,
    )


def verify_effect(service, action, receipt):
    duplicate = service.replay(action.idempotency_key)
    valid = receipt["delivery_address"] == APPROVED_NEW_ADDRESS and duplicate["duplicate"] is True
    return duplicate, valid


async def confirm(client, receipt):
    return await propose(
        client,
        Confirmation,
        f"Confirm only this verified receipt with status completed: {receipt}",
    )


def build_graph(request=SERVICE_REQUEST):
    client = create_client()
    service = SimulatedService.from_path(service_path)

    def inspect_state_node(state):
        order = inspect_state(service)
        return {
            **state,
            "request": request,
            "service": service,
            "trace": [{"event": "inspect_state", "order": order}],
            "model_calls": 0,
        }

    async def propose_action_node(state):
        action = await propose_action(client, state["request"])
        return {
            **state,
            "action": action,
            "model_calls": 1,
            "trace": [*state["trace"], {"event": "propose_action", "action": action.action}],
        }

    def authorise_action_node(state):
        authorised = authorise_action(state["action"])
        return {
            **state,
            "authorised": authorised,
            "trace": [
                *state["trace"],
                {"event": "authorise_action", "authorised": authorised},
            ],
        }

    def checkpoint_node(state):
        return {
            **state,
            "service": checkpoint(state["service"]),
            "trace": [*state["trace"], {"event": "checkpoint"}],
        }

    def execute_action_node(state):
        receipt = execute_action(state["service"], state["action"])
        return {
            **state,
            "receipt": receipt,
            "trace": [*state["trace"], {"event": "execute_action", "receipt": receipt}],
        }

    def verify_effect_node(state):
        duplicate, valid = verify_effect(state["service"], state["action"], state["receipt"])
        return {
            **state,
            "duplicate": duplicate,
            "effect_valid": valid,
            "trace": [*state["trace"], {"event": "verify_effect", "valid": valid}],
        }

    async def confirm_node(state):
        confirmation = await confirm(client, state["receipt"])
        return {
            **state,
            "confirmation": confirmation,
            "outcome": confirmation.status,
            "termination": "criteria_met",
            "model_calls": 2,
            "trace": [*state["trace"], {"event": "confirm", "status": confirmation.status}],
        }

    def refuse_node(state):
        termination = (
            "approval_mismatch"
            if not state.get("authorised", True)
            else "effect_verification_failed"
        )
        return {**state, "outcome": "refused", "termination": termination}

    graph = StateGraph(dict)
    for name, node in [
        ("inspect_state", inspect_state_node),
        ("propose_action", propose_action_node),
        ("authorise_action", authorise_action_node),
        ("checkpoint", checkpoint_node),
        ("execute_action", execute_action_node),
        ("verify_effect", verify_effect_node),
        ("confirm", confirm_node),
        ("refuse", refuse_node),
    ]:
        graph.add_node(name, node)

    graph.add_edge(START, "inspect_state")
    graph.add_edge("inspect_state", "propose_action")
    graph.add_edge("propose_action", "authorise_action")
    graph.add_conditional_edges(
        "authorise_action",
        lambda state: "checkpoint" if state["authorised"] else "refuse",
    )
    graph.add_edge("checkpoint", "execute_action")
    graph.add_edge("execute_action", "verify_effect")
    graph.add_conditional_edges(
        "verify_effect",
        lambda state: "confirm" if state["effect_valid"] else "refuse",
    )
    graph.add_edge("confirm", END)
    graph.add_edge("refuse", END)
    return graph.compile()


first = await build_graph().ainvoke({})
second = await build_graph().ainvoke({}) if MODEL_PROVIDER == "mock" else first
events = [item["event"] for item in first["trace"]]
evaluation = {
    "component": first["receipt"]["delivery_address"] == APPROVED_NEW_ADDRESS,
    "trajectory": events
    == [
        "inspect_state",
        "propose_action",
        "authorise_action",
        "checkpoint",
        "execute_action",
        "verify_effect",
        "confirm",
    ],
    "task": first["outcome"] == "completed",
    "safety": first["duplicate"]["duplicate"] is True,
    "repeated_run": first["trace"] == second["trace"] and first["receipt"] == second["receipt"],
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), {"evaluation": evaluation, "result": first}
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "termination": first["termination"],
    "resource_report": {"model_calls": first["model_calls"], "graph_nodes": 8},
}

## Limitation

The service, checkpoint and approval identity are simulated in process. Production systems require authenticated principals, durable state and transactional APIs.
